# 11. Evidence-Linked Action Items

## Goal

Extract explicit meeting assignments into structured action items with transcript evidence.

## Why This Step Matters

A meeting answer is useful, but an auditable task with an owner, deadline, and timestamp can support a real follow-up workflow. The extractor stays conservative: it records only explicit assignment phrases and leaves every item unapproved for human review.

## Inputs

Cleaned transcript rows containing `timestamp` and `text`.

## Outputs

Structured dictionaries containing `task`, `owner`, `deadline`, review state, and transcript evidence.

## Setup

In [6]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.summarization.action_items import extract_action_items

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/lataeq/Code/Saranote


## Load Data

In [7]:
transcript_rows = [
    {
        "timestamp": "00:00:21 - 00:00:29",
        "text": "ฝ่ายดำเนินงานต้องส่งเอกสารงบประมาณฉบับแก้ไขภายในวันศุกร์",
    },
    {
        "timestamp": "00:00:29 - 00:00:35",
        "text": "ที่ประชุมพูดถึงความเสี่ยงเรื่องข้อมูลไม่ครบ",
    },
    {
        "timestamp": "00:00:35 - 00:00:41",
        "text": "ที่ประชุมขอให้ทุกฝ่ายตรวจสอบข้อมูลก่อนส่งรายงาน",
    },
]

pd.DataFrame(transcript_rows)

,timestamp,text
0,00:00:21 - 00:00:29,ฝ่ายดำเนินงานต้องส่งเอกสารงบประมาณฉบับแก้ไขภาย...
1,00:00:29 - 00:00:35,ที่ประชุมพูดถึงความเสี่ยงเรื่องข้อมูลไม่ครบ
2,00:00:35 - 00:00:41,ที่ประชุมขอให้ทุกฝ่ายตรวจสอบข้อมูลก่อนส่งรายงาน


## Run Experiment

In [8]:
action_items = extract_action_items(transcript_rows)
print(json.dumps(action_items, ensure_ascii=False, indent=2))

[
  {
    "task": "ส่งเอกสารงบประมาณฉบับแก้ไข",
    "owner": "ฝ่ายดำเนินงาน",
    "deadline": "วันศุกร์",
    "status": "pending",
    "approved": false,
    "evidence": {
      "source": "transcript",
      "timestamp": "00:00:21 - 00:00:29",
      "text": "ฝ่ายดำเนินงานต้องส่งเอกสารงบประมาณฉบับแก้ไขภายในวันศุกร์"
    }
  },
  {
    "task": "ตรวจสอบข้อมูล",
    "owner": "ทุกฝ่าย",
    "deadline": "ก่อนส่งรายงาน",
    "status": "pending",
    "approved": false,
    "evidence": {
      "source": "transcript",
      "timestamp": "00:00:35 - 00:00:41",
      "text": "ที่ประชุมขอให้ทุกฝ่ายตรวจสอบข้อมูลก่อนส่งรายงาน"
    }
  }
]


## Inspect Results

In [9]:
action_items_df = pd.DataFrame(
    [
        {
            "task": item["task"],
            "owner": item["owner"],
            "deadline": item["deadline"],
            "status": item["status"],
            "approved": item["approved"],
            "timestamp": item["evidence"]["timestamp"],
        }
        for item in action_items
    ]
)
action_items_df

,task,owner,deadline,status,approved,timestamp
0,ส่งเอกสารงบประมาณฉบับแก้ไข,ฝ่ายดำเนินงาน,วันศุกร์,pending,False,00:00:21 - 00:00:29
1,ตรวจสอบข้อมูล,ทุกฝ่าย,ก่อนส่งรายงาน,pending,False,00:00:35 - 00:00:41


## Checks

In [10]:
checks = [
    ("finds two actions", len(action_items) == 2),
    ("extracts explicit owner/task", action_items[0]["owner"] == "ฝ่ายดำเนินงาน"),
    ("keeps timestamps", action_items[0]["evidence"]["timestamp"] == transcript_rows[0]["timestamp"]),
    ("requires approval before send", all(not item["approved"] for item in action_items)),
    ("ignores general discussion", all("ความเสี่ยง" not in item["task"] for item in action_items)),
]
checks_df = pd.DataFrame(checks, columns=["check", "passed"])
print(checks_df.to_string(index=False))
assert checks_df["passed"].all(), "Some action-item checks failed"
print(f"Passed {checks_df['passed'].sum()}/{len(checks_df)} action-item checks")

                        check  passed
            finds two actions    True
 extracts explicit owner/task    True
             keeps timestamps    True
requires approval before send    True
   ignores general discussion    True
Passed 5/5 action-item checks


## What We Learned

- Explicit assignments can become structured tasks without a generative model.
- Each task keeps the exact transcript text and timestamp that support it.
- New tasks remain unapproved, creating a safe boundary before future automation.

## Common Problems

- Indirect assignments without `ต้อง` or `ขอให้ทุกฝ่าย` are intentionally not extracted yet.
- ASR errors in owner names, action phrases, or deadlines can still prevent extraction.
- Missing deadlines remain `None`; the system does not invent one.

## Next Step

Run this extractor on more real Thai meeting transcripts. After its false-positive rate is acceptable, expose approved action items to an n8n webhook.